# Resumo do código

Este notebook resolve o exercício do cronograma de estudos do pequeno Turing como um problema da **Mochila Binária (0/1)**. Cada disciplina é tratada como um item indivisível: ou Turing estuda o tempo necessário e obtém toda a importância subjetiva, ou não escolhe a disciplina. O objetivo é maximizar a importância subjetiva total respeitando o limite de $16$ horas.

Enunciado do exercício: [exercicio_1.pdf](./exercicio_1.pdf).  

A recorrência usada é:

$$
M[i][h] = \begin{cases}
M[i-1][h], & \text{se } t_i > h \\
\max(M[i-1][h],\; is_i + M[i-1][h-t_i]), & \text{se } t_i \le h
\end{cases}
$$

em que $M[i][h]$ é a melhor importância subjetiva usando as $i$ primeiras disciplinas e até $h$ horas disponíveis.

```mermaid
flowchart TD
    A[Lista de disciplinas] --> B[Montar tabela M]
    B --> C{Para cada disciplina e capacidade}
    C --> D[Comparar não escolher vs. escolher]
    D --> E[Guardar melhor valor em M]
    E --> F[Reconstruir cronograma pela tabela]
    F --> G[Mostrar tabela, solução e estatísticas]
```

Células do notebook:

1. **Importações e modelo de dados**: carrega bibliotecas padrão e define a estrutura `Disciplina`.
2. **Dados do exercício**: registra as disciplinas, tempos, importâncias subjetivas e o limite de horas.
3. **Programação dinâmica**: constrói a tabela $M$ da mochila binária.
4. **Reconstrução da solução**: recupera quais disciplinas formam o melhor cronograma.
5. **Formatação didática**: prepara tabelas em Markdown para visualização no notebook.
6. **Execução da solução**: mostra a tabela usada, o cronograma escolhido e o resumo da resposta.
7. **Conferência por força bruta**: valida a solução testando todas as combinações possíveis.
8. **Estatísticas e complexidade**: mede tempo, memória e apresenta as ordens de complexidade.

In [1]:
# Esta célula importa apenas bibliotecas padrão e define o modelo de dados usado no problema.
from dataclasses import dataclass
from itertools import combinations
from time import perf_counter_ns, process_time_ns
import tracemalloc
from IPython.display import Markdown, display


@dataclass(frozen=True)
class Disciplina:
    """Representa uma disciplina candidata ao cronograma de estudos."""

    nome: str
    tempo: int
    importancia: int

In [2]:
# Esta célula registra os dados do enunciado: capacidade de estudo e lista de disciplinas.
TEMPO_DISPONIVEL = 16

DISCIPLINAS = [
    Disciplina("ALP", 3, 5),
    Disciplina("PAP", 5, 3),
    Disciplina("CAL", 6, 6),
    Disciplina("BAN", 4, 4),
    Disciplina("REC", 9, 5),
    Disciplina("PPR", 3, 2),
    Disciplina("TEC", 8, 6),
    Disciplina("SOP", 4, 4),
]

print(f"Tempo disponível: {TEMPO_DISPONIVEL} horas")
print(f"Quantidade de disciplinas: {len(DISCIPLINAS)}")

Tempo disponível: 16 horas
Quantidade de disciplinas: 8


## Estratégia

A abordagem gulosa por maior importância, menor tempo ou melhor razão $is/tempo$ não garante a solução ótima para a mochila binária. Por isso, usamos programação dinâmica: cada célula da tabela guarda a melhor resposta já conhecida para um subproblema menor.

In [3]:
# Esta célula constrói a tabela de programação dinâmica da Mochila Binária (0/1).
def construir_tabela_mochila(disciplinas: list[Disciplina], capacidade: int) -> list[list[int]]:
    """Retorna a tabela M em que M[i][h] é a melhor importância com i disciplinas e h horas."""
    quantidade = len(disciplinas)
    tabela = [[0] * (capacidade + 1) for _ in range(quantidade + 1)]

    for i, disciplina in enumerate(disciplinas, start=1):
        for horas in range(capacidade + 1):
            sem_disciplina = tabela[i - 1][horas]

            if disciplina.tempo <= horas:
                com_disciplina = disciplina.importancia + tabela[i - 1][horas - disciplina.tempo]
                tabela[i][horas] = max(sem_disciplina, com_disciplina)
            else:
                tabela[i][horas] = sem_disciplina

    return tabela

In [4]:
# Esta célula reconstrói o cronograma ótimo caminhando de trás para frente na tabela.
def reconstruir_cronograma(
    disciplinas: list[Disciplina],
    capacidade: int,
    tabela: list[list[int]],
) -> list[Disciplina]:
    """Recupera as disciplinas escolhidas a partir da tabela M já preenchida."""
    escolhidas = []
    horas_restantes = capacidade

    for i in range(len(disciplinas), 0, -1):
        if tabela[i][horas_restantes] != tabela[i - 1][horas_restantes]:
            disciplina = disciplinas[i - 1]
            escolhidas.append(disciplina)
            horas_restantes -= disciplina.tempo

    escolhidas.reverse()
    return escolhidas

In [5]:
# Esta célula contém funções auxiliares para exibir os dados e a tabela em Markdown.
def tabela_disciplinas_markdown(disciplinas: list[Disciplina]) -> str:
    linhas = ["| Disciplina | Tempo (h) | Importância subjetiva (is) |", "|---:|---:|---:|"]
    for disciplina in disciplinas:
        linhas.append(f"| {disciplina.nome} | {disciplina.tempo} | {disciplina.importancia} |")
    return "\n".join(linhas)


def tabela_dp_markdown(disciplinas: list[Disciplina], tabela: list[list[int]]) -> str:
    cabecalho = ["Item / capacidade"] + [str(horas) for horas in range(len(tabela[0]))]
    alinhamento = ["---"] + ["---:"] * (len(cabecalho) - 1)
    linhas = ["| " + " | ".join(cabecalho) + " |", "| " + " | ".join(alinhamento) + " |"]

    linhas.append("| 0 disciplinas | " + " | ".join(map(str, tabela[0])) + " |")
    for i, disciplina in enumerate(disciplinas, start=1):
        rotulo = f"{i} - {disciplina.nome}"
        linhas.append("| " + rotulo + " | " + " | ".join(map(str, tabela[i])) + " |")

    return "\n".join(linhas)


def cronograma_markdown(escolhidas: list[Disciplina]) -> str:
    linhas = ["| Ordem | Disciplina | Tempo (h) | Importância subjetiva (is) |", "|---:|---:|---:|---:|"]
    for ordem, disciplina in enumerate(escolhidas, start=1):
        linhas.append(f"| {ordem} | {disciplina.nome} | {disciplina.tempo} | {disciplina.importancia} |")
    return "\n".join(linhas)


In [6]:
# Esta célula executa a solução, mostra a tabela usada e apresenta o melhor cronograma.
tabela = construir_tabela_mochila(DISCIPLINAS, TEMPO_DISPONIVEL)
cronograma_otimo = reconstruir_cronograma(DISCIPLINAS, TEMPO_DISPONIVEL, tabela)

tempo_total = sum(disciplina.tempo for disciplina in cronograma_otimo)
importancia_total = sum(disciplina.importancia for disciplina in cronograma_otimo)

relatorio = f"""
## Dados do exercício

{tabela_disciplinas_markdown(DISCIPLINAS)}

## Tabela usada na solução

Cada linha representa as primeiras disciplinas consideradas; cada coluna representa uma capacidade de estudo de $0$ a ${TEMPO_DISPONIVEL}$ horas.

{tabela_dp_markdown(DISCIPLINAS, tabela)}

## Melhor cronograma encontrado

{cronograma_markdown(cronograma_otimo)}

**Resposta:** estudar **{', '.join(d.nome for d in cronograma_otimo)}**, usando **{tempo_total} horas** e obtendo importância subjetiva total **{importancia_total} is**.
"""

display(Markdown(relatorio))



## Dados do exercício

| Disciplina | Tempo (h) | Importância subjetiva (is) |
|---:|---:|---:|
| ALP | 3 | 5 |
| PAP | 5 | 3 |
| CAL | 6 | 6 |
| BAN | 4 | 4 |
| REC | 9 | 5 |
| PPR | 3 | 2 |
| TEC | 8 | 6 |
| SOP | 4 | 4 |

## Tabela usada na solução

Cada linha representa as primeiras disciplinas consideradas; cada coluna representa uma capacidade de estudo de $0$ a $16$ horas.

| Item / capacidade | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 | 13 | 14 | 15 | 16 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 0 disciplinas | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 1 - ALP | 0 | 0 | 0 | 5 | 5 | 5 | 5 | 5 | 5 | 5 | 5 | 5 | 5 | 5 | 5 | 5 | 5 |
| 2 - PAP | 0 | 0 | 0 | 5 | 5 | 5 | 5 | 5 | 8 | 8 | 8 | 8 | 8 | 8 | 8 | 8 | 8 |
| 3 - CAL | 0 | 0 | 0 | 5 | 5 | 5 | 6 | 6 | 8 | 11 | 11 | 11 | 11 | 11 | 14 | 14 | 14 |
| 4 - BAN | 0 | 0 | 0 | 5 | 5 | 5 | 6 | 9 | 9 | 11 | 11 | 11 | 12 | 15 | 15 | 15 | 15 |
| 5 - REC | 0 | 0 | 0 | 5 | 5 | 5 | 6 | 9 | 9 | 11 | 11 | 11 | 12 | 15 | 15 | 15 | 15 |
| 6 - PPR | 0 | 0 | 0 | 5 | 5 | 5 | 7 | 9 | 9 | 11 | 11 | 11 | 13 | 15 | 15 | 15 | 17 |
| 7 - TEC | 0 | 0 | 0 | 5 | 5 | 5 | 7 | 9 | 9 | 11 | 11 | 11 | 13 | 15 | 15 | 15 | 17 |
| 8 - SOP | 0 | 0 | 0 | 5 | 5 | 5 | 7 | 9 | 9 | 11 | 11 | 13 | 13 | 15 | 15 | 15 | 17 |

## Melhor cronograma encontrado

| Ordem | Disciplina | Tempo (h) | Importância subjetiva (is) |
|---:|---:|---:|---:|
| 1 | ALP | 3 | 5 |
| 2 | CAL | 6 | 6 |
| 3 | BAN | 4 | 4 |
| 4 | PPR | 3 | 2 |

**Resposta:** estudar **ALP, CAL, BAN, PPR**, usando **16 horas** e obtendo importância subjetiva total **17 is**.


In [7]:
# Esta célula confere a solução por força bruta, testando todas as combinações de disciplinas.
def resolver_por_forca_bruta(
    disciplinas: list[Disciplina],
    capacidade: int,
) -> tuple[int, list[tuple[int, tuple[Disciplina, ...]]]]:
    """Retorna o valor ótimo e todas as combinações ótimas encontradas por enumeração completa."""
    melhor_importancia = 0
    combinacoes_otimas = []

    for tamanho in range(len(disciplinas) + 1):
        for combinacao in combinations(disciplinas, tamanho):
            tempo = sum(disciplina.tempo for disciplina in combinacao)
            importancia = sum(disciplina.importancia for disciplina in combinacao)

            if tempo <= capacidade and importancia > melhor_importancia:
                melhor_importancia = importancia
                combinacoes_otimas = [(tempo, combinacao)]
            elif tempo <= capacidade and importancia == melhor_importancia:
                combinacoes_otimas.append((tempo, combinacao))

    return melhor_importancia, combinacoes_otimas


importancia_bruta, combinacoes_otimas = resolver_por_forca_bruta(DISCIPLINAS, TEMPO_DISPONIVEL)
nomes_cronograma = tuple(d.nome for d in cronograma_otimo)
nomes_otimos = [tuple(d.nome for d in combinacao) for _, combinacao in combinacoes_otimas]

assert importancia_bruta == importancia_total
assert nomes_cronograma in nomes_otimos

print("Conferência por força bruta concluída com sucesso.")
print(f"Importância subjetiva ótima: {importancia_bruta} is")
print("Combinações ótimas encontradas:")
for tempo, combinacao in combinacoes_otimas:
    nomes = ", ".join(d.nome for d in combinacao)
    print(f"- {nomes}: {tempo} horas")


Conferência por força bruta concluída com sucesso.
Importância subjetiva ótima: 17 is
Combinações ótimas encontradas:
- ALP, CAL, BAN, PPR: 16 horas
- ALP, CAL, PPR, SOP: 16 horas


In [8]:
# Esta célula mede tempo e memória da solução e informa as ordens de complexidade.
def medir_execucao(disciplinas: list[Disciplina], capacidade: int) -> dict[str, float | int]:
    """Executa a solução uma vez e coleta métricas de processamento e memória."""
    tracemalloc.start()
    inicio_parede = perf_counter_ns()
    inicio_cpu = process_time_ns()

    tabela_medida = construir_tabela_mochila(disciplinas, capacidade)
    reconstruir_cronograma(disciplinas, capacidade, tabela_medida)

    fim_cpu = process_time_ns()
    fim_parede = perf_counter_ns()
    memoria_atual, pico_memoria = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return {
        "tempo_parede_ms": (fim_parede - inicio_parede) / 1_000_000,
        "tempo_cpu_ms": (fim_cpu - inicio_cpu) / 1_000_000,
        "memoria_atual_bytes": memoria_atual,
        "pico_memoria_bytes": pico_memoria,
        "celulas_tabela": (len(disciplinas) + 1) * (capacidade + 1),
    }


estatisticas = medir_execucao(DISCIPLINAS, TEMPO_DISPONIVEL)
quantidade = len(DISCIPLINAS)
capacidade = TEMPO_DISPONIVEL

estatisticas_markdown = fr"""
## Estatísticas de execução

| Métrica | Valor |
|---|---:|
| Disciplinas ($n$) | {quantidade} |
| Capacidade ($C$) | {capacidade} horas |
| Células da tabela $(n+1)(C+1)$ | {estatisticas['celulas_tabela']} |
| Tempo de parede | {estatisticas['tempo_parede_ms']:.4f} ms |
| Tempo de CPU | {estatisticas['tempo_cpu_ms']:.4f} ms |
| Memória atual rastreada | {estatisticas['memoria_atual_bytes']} bytes |
| Pico de memória rastreado | {estatisticas['pico_memoria_bytes']} bytes |

## Ordem de complexidade

- **Tempo:** $\Theta(n \cdot C)$, pois cada disciplina é avaliada para cada capacidade inteira de $0$ até $C$.
- **Memória:** $\Theta(n \cdot C)$, pois a solução didática mantém a tabela completa para mostrar a solução e reconstruir o cronograma.

Para este exercício, $n = {quantidade}$ e $C = {capacidade}$, então a tabela possui ${quantidade + 1} \times {capacidade + 1} = {estatisticas['celulas_tabela']}$ células.
"""

display(Markdown(estatisticas_markdown))



## Estatísticas de execução

| Métrica | Valor |
|---|---:|
| Disciplinas ($n$) | 8 |
| Capacidade ($C$) | 16 horas |
| Células da tabela $(n+1)(C+1)$ | 153 |
| Tempo de parede | 0.0447 ms |
| Tempo de CPU | 0.0390 ms |
| Memória atual rastreada | 1416 bytes |
| Pico de memória rastreado | 1456 bytes |

## Ordem de complexidade

- **Tempo:** $\Theta(n \cdot C)$, pois cada disciplina é avaliada para cada capacidade inteira de $0$ até $C$.
- **Memória:** $\Theta(n \cdot C)$, pois a solução didática mantém a tabela completa para mostrar a solução e reconstruir o cronograma.

Para este exercício, $n = 8$ e $C = 16$, então a tabela possui $9 \times 17 = 153$ células.
